# Step 5 – Environmental optimisation (CO₂ costs)
**MJ2505 Project 2 – District heating network optimisation**

| | |
|---|---|
| **Goal** | Add **CO₂ emission costs** to every unit's variable cost and compare dispatch, emissions and cost in three cases: **Step 4 (economic)**, **Scenario A – cost + emissions**, **Scenario B – emissions only** (all fuel/operating costs = 0). Plus at least one **sensitivity**. |
| **Reads** | `data/step4/oemof_input_with_losses.csv` · `data/step3/technology_parameters.csv`, `carriers.csv` |
| **Writes** | `results/step5/` |
| **Grading** | A – reformulate dispatch costs with CO₂, solve both scenarios, compare on economic **and** environmental measures (trade-off, not a single "best"), sensitivity analysis, limitations |

Same network, same adjusted demand (incl. DHNx losses), same five units and capacity definitions – **the only change is the emission cost**. Legend: 🔧 ASSUMPTION · ✏️ TODO

## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from helpers import (DATA, RESULTS, INTEREST_RATE, build_and_solve_heat_system, summarize_units,
                     heat_dispatch, plot_dispatch, UNIT_COLORS)

OUT = RESULTS / "step5"
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Diagnostic (from the course "Prerequisites" document): which Python / packages / solver are used?
import sys, shutil
import pandas as pd
import oemof.solph as solph
import dhnx
from helpers import SOLVER

print("Python executable:", sys.executable)
print("Python version:   ", sys.version.split()[0])
print("DHNx version:     ", dhnx.__version__)
print("oemof.solph:      ", solph.__version__)
print("pandas:           ", pd.__version__)
print("Solver used:      ", SOLVER, "->", shutil.which("cbc") or "HiGHS via the 'highspy' pip package")

## 1. Inputs from Step 3 and Step 4

In [ ]:
data = pd.read_csv(DATA / "step4" / "oemof_input_with_losses.csv", index_col="date", parse_dates=True).asfreq("h")
tech = pd.read_csv(DATA / "step3" / "technology_parameters.csv", index_col="unit")
carriers = pd.read_csv(DATA / "step3" / "carriers.csv", index_col="carrier")
DEMAND = "heat demand incl. losses"
print(f"Annual heat to generate: {data[DEMAND].sum():,.1f} MWh")
carriers

## 2. CO₂ price and emission cost per unit

🔧 **ASSUMPTIONS:** CO₂ price (e.g. EU ETS level for your year – cite) and the emission factors per energy carrier (defined in Step 3, `carriers.csv`: natural gas ≈ 0.202 t/MWh, biomass counted as 0 (biogenic), grid electricity = ✏️ country average or marginal factor).

**Reformulated variable cost of each unit** (objective function itself unchanged – oemof still minimises total cost):
$$c^{var}_{u} = c^{O\&M}_u + p_{CO_2}\cdot\frac{EF_{in(u)}}{\eta_u} \quad [\text{€/MWh}_{heat}]$$
and the fuel cost $p^{fuel}_{u,t}$ stays on the fuel source. In `build_and_solve_heat_system()` this is the `var_cost()` function, activated by the argument `co2_price`.

In [ ]:
CO2_PRICE = 80.0     # 🔧 EUR/tCO2  (placeholder - cite EU ETS price for your reference year)

emission_cost = pd.DataFrame({
    "input": tech["input"],
    "EF input [t/MWh_in]": tech["input"].map(carriers["ef_tCO2_per_MWh"]),
    "efficiency": tech["efficiency"],
})
emission_cost["CO2 [t/MWh_heat]"] = emission_cost["EF input [t/MWh_in]"] / emission_cost["efficiency"]
emission_cost["CO2 cost [EUR/MWh_heat]"] = CO2_PRICE * emission_cost["CO2 [t/MWh_heat]"]
emission_cost["var O&M [EUR/MWh_heat]"] = tech["var_om_eur_per_MWh"]
emission_cost.round(3)

## 3. Run the three cases
| Case | fuel/electricity cost | var O&M | CO₂ cost | investment cost |
|---|---|---|---|---|
| Step 4 – economic | ✓ | ✓ | – | ✓ |
| **A – cost + emissions** | ✓ | ✓ | ✓ | ✓ |
| **B – emissions only** | 0 | 0 | ✓ | ✓ (capacity definitions kept) |

✏️ Discuss in the report: in B, investment costs are kept so that capacities stay defined; with only CO₂ costs, several zero-emission units are equally cheap to operate, so the dispatch among them may not be unique.

In [ ]:
scenarios = {
    "Step 4 (economic)":  dict(co2_price=0.0),
    "A: cost + CO2":      dict(co2_price=CO2_PRICE),
    "B: CO2 only":        dict(co2_price=CO2_PRICE, include_fuel_costs=False, include_var_om=False),
}
summaries, dispatches = {}, {}
for name, kw in scenarios.items():
    model, results, parts = build_and_solve_heat_system(data, tech, carriers, demand_col=DEMAND, i=INTEREST_RATE, **kw)
    summaries[name] = summarize_units(results, parts, tech, carriers)
    dispatches[name] = heat_dispatch(results, parts)
    print(f"{name:20s} objective = {model.objective():>12,.0f} EUR/a")
units = list(tech.index)

## 4. Compare the scenarios

### 4.1 Generation mix and capacities

In [ ]:
share = pd.DataFrame({n: s.loc[units, "share of heat [%]"] for n, s in summaries.items()})
capacity = pd.DataFrame({n: s.loc[units, "capacity [MW]"] for n, s in summaries.items()})
display(pd.concat({"share of heat [%]": share, "capacity [MW]": capacity}, axis=1).round(1))

share.T.plot.bar(stacked=True, figsize=(9, 4), color=[UNIT_COLORS[u] for u in units], title="Heat generation mix")
plt.ylabel("% of heat"); plt.legend(bbox_to_anchor=(1.01, 1)); plt.tight_layout()
plt.savefig(OUT / "generation_mix.png", dpi=150); plt.show()

### 4.2 Economic and environmental indicators
To compare on the **same economic basis**, all costs are evaluated with real prices (`summarize_units` always uses the real fuel prices, even for scenario B where the optimiser ignored them):
- **System cost excl. CO₂** = capex annuity + fuel/electricity + O&M (what the heat really costs to produce)
- **CO₂ cost** = emissions × CO₂ price (same price for all cases, to show the "carbon bill")

In [ ]:
def indicators(s):
    sys_cost = s.loc["TOTAL", ["annualised capex [EUR]", "fuel/electricity cost [EUR]", "var O&M cost [EUR]"]].sum()
    co2 = s.loc["TOTAL", "CO2 [t]"]
    heat = data[DEMAND].sum()
    return pd.Series({
        "CO2 emissions [t/a]": co2,
        "CO2 intensity [kg/MWh]": 1000 * co2 / heat,
        "system cost excl. CO2 [EUR/a]": sys_cost,
        "CO2 cost at CO2 price [EUR/a]": co2 * CO2_PRICE,
        "system cost incl. CO2 [EUR/a]": sys_cost + co2 * CO2_PRICE,
        "cost of heat excl. CO2 [EUR/MWh]": sys_cost / heat,
    })

kpi = pd.DataFrame({n: indicators(s) for n, s in summaries.items()})
ref = "Step 4 (economic)"
for n in kpi.columns.drop(ref):
    kpi[f"Δ {n} vs Step 4 [%]"] = 100 * (kpi[n] / kpi[ref] - 1)
kpi.round(2)

### 4.3 Trade-off between cost and emissions
Each case is one point; the slope between two points is the **CO₂ abatement cost** (€/t avoided).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for n in summaries:
    ax.scatter(kpi.loc["CO2 emissions [t/a]", n], kpi.loc["system cost excl. CO2 [EUR/a]", n], s=80, label=n)
ax.set_xlabel("CO2 emissions [t/a]"); ax.set_ylabel("System cost excl. CO2 [EUR/a]")
ax.set_title("Economic vs. environmental outcome"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / "tradeoff.png", dpi=150); plt.show()

for n in ["A: cost + CO2", "B: CO2 only"]:
    d_cost = kpi.loc["system cost excl. CO2 [EUR/a]", n] - kpi.loc["system cost excl. CO2 [EUR/a]", ref]
    d_co2 = kpi.loc["CO2 emissions [t/a]", ref] - kpi.loc["CO2 emissions [t/a]", n]
    print(f"{n}: abatement cost = {d_cost / d_co2:,.1f} EUR/t avoided" if d_co2 > 1e-6 else f"{n}: no CO2 reduction")

In [ ]:
for n, d in dispatches.items():
    plot_dispatch(d, f"Heat production – {n}")
    plt.savefig(OUT / f"dispatch_{n.split(':')[0].replace(' ', '_')}.png", dpi=150); plt.show()

## 5. Sensitivity analysis (required for grade A)
Vary **one** uncertain parameter and show how the result changes. Placeholder: the **CO₂ price** in scenario A.
✏️ Choose and justify your parameter(s) – other good candidates: electricity price level, gas price, heat-pump COP or max capacity, grid emission factor, network loss (Step 4).

In [ ]:
CO2_PRICES = [0, 25, 50, 80, 120, 200, 300]     # 🔧 EUR/t
rows = []
for p in CO2_PRICES:
    _, res_p, parts_p = build_and_solve_heat_system(data, tech, carriers, demand_col=DEMAND, i=INTEREST_RATE, co2_price=p)
    s = summarize_units(res_p, parts_p, tech, carriers)
    row = {"CO2 price [EUR/t]": p,
           "CO2 [t/a]": s.loc["TOTAL", "CO2 [t]"],
           "system cost excl. CO2 [EUR/a]": s.loc["TOTAL", ["annualised capex [EUR]", "fuel/electricity cost [EUR]", "var O&M cost [EUR]"]].sum()}
    row.update({f"share {u} [%]": s.loc[u, "share of heat [%]"] for u in units})
    rows.append(row)
sensitivity = pd.DataFrame(rows).set_index("CO2 price [EUR/t]")

fig, axs = plt.subplots(1, 2, figsize=(14, 4))
sensitivity[[f"share {u} [%]" for u in units]].plot.area(ax=axs[0], color=[UNIT_COLORS[u] for u in units],
                                                         title="Generation mix vs. CO2 price (scenario A)")
axs[0].set_ylabel("% of heat"); axs[0].legend([u for u in units], fontsize=8)
ax2 = axs[1].twinx()
sensitivity["CO2 [t/a]"].plot(ax=axs[1], marker="o", color="tab:green", title="Emissions and cost vs. CO2 price")
sensitivity["system cost excl. CO2 [EUR/a]"].plot(ax=ax2, marker="s", color="tab:blue")
axs[1].set_ylabel("CO2 [t/a]", color="tab:green"); ax2.set_ylabel("system cost excl. CO2 [EUR/a]", color="tab:blue")
plt.tight_layout(); plt.savefig(OUT / "sensitivity_co2_price.png", dpi=150); plt.show()
sensitivity.round(1)

## 6. Limitations of the network and energy-system representation
✏️ Discuss (required for grade A). Starting points:
- DHNx designs for **one peak hour** (`heat_demand='scalar'`); operating costs approximated with full-load hours; losses flow-independent and constant over the year.
- No hydraulics / pressure drop / pumping energy; DN capacity from a velocity rule of thumb.
- oemof: **perfect foresight**, linear model, no part-load efficiencies, start-up costs or minimum load; constant COP (no dependence on source temperature).
- **No thermal storage** (by assignment) – storage would change dispatch, especially for solar and electricity-price-driven units.
- Placeholder price data / synthetic solar profile; emission accounting conventions (biogenic CO₂ = 0, average vs. marginal grid factor).
- The network (DHNx) and supply (oemof) are optimised **sequentially** (soft link), not jointly.

## 7. Save results

In [ ]:
kpi.to_csv(OUT / "scenario_indicators.csv")
pd.concat(summaries).to_csv(OUT / "unit_summaries_all_scenarios.csv")
sensitivity.to_csv(OUT / "sensitivity_co2_price.csv")
emission_cost.to_csv(OUT / "emission_cost_per_unit.csv")
print("Saved to", OUT)

## 8. Questions to answer
✏️

1. **Scenario A: which units become less competitive when emission costs are included?** *(sections 2, 4.1 – compare with Step 4)*
2. **Scenario B: which low-carbon units dominate operation?** *(4.1)*
3. **What is the trade-off between economic and environmental outcomes (Step 4 vs. A vs. B)?** *(4.2, 4.3 – characterise the trade-off, don't just pick a "best")*
4. Sensitivity (section 5) and limitations (section 6).